<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-17-March-31-2026/Lecture-17_NeuralNetworks-FunctionFitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Define target function

def target_fn(x: torch.Tensor) -> torch.Tensor:
    """A non-trivial 1-D function to approximate."""
    return torch.sin(2 * np.pi * x) + 0.5 * torch.sin(5 * np.pi * x)

# Sample training data randomly from a uniform distribution in x.
# The training data will have some random noise in it.
# We also generate the real function on a dense grid for comparision

N_TRAIN   = 100
N_TEST    = 2000
X_MIN, X_MAX = -1.5, 1.5
NOISE_STD = 0.20

x_train = torch.rand(N_TRAIN, 1) * (X_MAX - X_MIN) + X_MIN   # uniform random
y_train = target_fn(x_train) + torch.randn_like(x_train) * NOISE_STD

x_test  = torch.linspace(X_MIN, X_MAX, N_TEST).unsqueeze(1)  # dense grid
y_test  = target_fn(x_test)

plt.plot(x_train.numpy(), y_train.numpy(), "o", label="Training samples")
plt.plot(x_test.numpy(), y_test.numpy(), "k", linewidth=2, label="True function")
plt.legend()
plt.xlabel("x");
plt.ylabel("f(x)")
plt.show()

In [ ]:
# Define a class for the neural network

class DeepNet(nn.Module):
    def __init__(self, hidden_dim: int = 128, n_layers: int = 6):
        super().__init__()
        layers = [nn.Linear(1, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

        # Xavier init for faster convergence
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

In [ ]:
# initialize the neural network

model = DeepNet(hidden_dim=128, n_layers=6)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}\n")



In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_parameters(model)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

In [ ]:
def count_weights_biases(model):
    weights, biases = 0, 0
    for name, param in model.named_parameters():
        if 'weight' in name:
            weights += param.numel()
        elif 'bias' in name:
            biases += param.numel()
    return weights, biases

weights, biases = count_weights_biases(model)
print(f"Weights: {weights:,}")
print(f"Biases:  {biases:,}")

In [ ]:
# We now train the network to fit the function from the training data
# We minimize the loss function given by the mean square error.
# We also use an optional regularization loss
# We use the Adam optimizer
#
# NOTE: If you want to run this again for different setting,
# you must re-initialize the neural network above, otherwise you will be starting
# from the previous NN.

EPOCHS    = 10_000
LR        = 1e-3
SCHEDULER_PATIENCE = 300

# ── Regularization config ─────────────────────────────────────────────────────
# Set REG_TYPE to: None | "L1" | "L2" | "ElasticNet"
REG_TYPE   = None      # None | "L1" | "L2" | "ElasticNet"
LAMBDA_L1  = 1e-4       # L1 penalty weight  (used for L1 and ElasticNet)
LAMBDA_L2  = 1e-4       # L2 penalty weight  (used for L2 and ElasticNet)


def regularization_loss(model: nn.Module, reg_type: str | None) -> torch.Tensor:
    """Return the regularization penalty for all weight tensors (biases excluded)."""
    if reg_type is None:
        return torch.tensor(0.0)

    weights = [p for name, p in model.named_parameters() if "weight" in name]

    if reg_type == "L1":
        return LAMBDA_L1 * sum(p.abs().sum() for p in weights)

    if reg_type == "L2":
        return LAMBDA_L2 * sum(p.pow(2).sum() for p in weights)

    if reg_type == "ElasticNet":
        l1 = LAMBDA_L1 * sum(p.abs().sum() for p in weights)
        l2 = LAMBDA_L2 * sum(p.pow(2).sum() for p in weights)
        return l1 + l2

    raise ValueError(f"Unknown reg_type '{reg_type}'. Choose None, 'L1', 'L2', or 'ElasticNet'.")


reg_label = REG_TYPE if REG_TYPE else "None"
print(f"Regularization : {reg_label}")
if REG_TYPE in ("L1", "ElasticNet"):
    print(f"  λ_L1 = {LAMBDA_L1}")
if REG_TYPE in ("L2", "ElasticNet"):
    print(f"  λ_L2 = {LAMBDA_L2}")
print()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=SCHEDULER_PATIENCE, factor=0.5
)

loss_history      = []   # total loss  (MSE + penalty)
mse_history       = []   # data term only
reg_loss_history  = []   # penalty term only

for epoch in range(1, EPOCHS + 1):
    # set the NN to training mode
    model.train()
    # set gradient to zero
    optimizer.zero_grad()

    # calculate the loss
    pred     = model(x_train)
    mse_loss = criterion(pred, y_train)
    reg_loss = regularization_loss(model, REG_TYPE)
    loss     = mse_loss + reg_loss

    # calculate the gradient of the loss
    loss.backward()
    # update the parameters based on the loss
    optimizer.step()
    scheduler.step(loss)

    #
    loss_history.append(loss.item())
    mse_history.append(mse_loss.item())
    reg_loss_history.append(reg_loss.item())

    if epoch % 500 == 0:
        print(f"Epoch {epoch:5d}/{EPOCHS} | Total: {loss.item():.6f} | "
              f"MSE: {mse_loss.item():.6f} | Reg ({reg_label}): {reg_loss.item():.6f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")




In [ ]:
# calculate the fit on a dense grid

model.eval()
with torch.no_grad():
    y_pred   = model(x_test)

test_mse = criterion(y_pred, y_test).item()
print(f"\nTest MSE: {test_mse:.6f}")




In [ ]:
# make a plot of the results.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# — Left: fit quality —
ax = axes[0]
ax.scatter(x_train.numpy(), y_train.numpy(),
           s=6, alpha=0.4, color="steelblue", label="Training samples")
ax.plot(x_test.numpy(), y_test.numpy(),
        color="black", linewidth=2, label="True function")
ax.plot(x_test.numpy(), y_pred.numpy(),
        color="crimson", linewidth=2, linestyle="--", label="NN prediction")
ax.set_title(f"1-D Function Fitting  [reg={reg_label}]", fontsize=13)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend()

# — Middle: total loss + MSE —
ax = axes[1]
ax.semilogy(loss_history, color="darkorange",   linewidth=1.5, label="Total loss (MSE + reg)")
ax.semilogy(mse_history,  color="steelblue",    linewidth=1.5, linestyle="--", label="MSE only")
ax.set_title("Training Loss (log scale)", fontsize=13)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, which="both", ls=":")

# — Right: regularization penalty —
ax = axes[2]
ax.semilogy(reg_loss_history, color="mediumseagreen", linewidth=1.5,
            label=f"{reg_label} penalty")
ax.set_title(f"Regularization Penalty  [λ_L1={LAMBDA_L1}, λ_L2={LAMBDA_L2}]", fontsize=13)
ax.set_xlabel("Epoch"); ax.set_ylabel("Penalty")
ax.legend()
ax.grid(True, which="both", ls=":")

plt.tight_layout()
# plt.savefig("fit_result.png", dpi=150)
# print("\nPlot saved to fit_result.png")
plt.show()


In [ ]:

def do_training(REG_TYPE=None):

  model = DeepNet(hidden_dim=128, n_layers=6)

  if REG_TYPE.upper() == "OFF": REG_TYPE=None

  EPOCHS    = 10_000
  LR        = 1e-3
  SCHEDULER_PATIENCE = 300

  # ── Regularization config ─────────────────────────────────────────────────────
  LAMBDA_L1  = 1e-4       # L1 penalty weight  (used for L1 and ElasticNet)
  LAMBDA_L2  = 1e-4       # L2 penalty weight  (used for L2 and ElasticNet)


  def regularization_loss(model: nn.Module, reg_type: str | None) -> torch.Tensor:
      """Return the regularization penalty for all weight tensors (biases excluded)."""
      if reg_type is None:
          return torch.tensor(0.0)

      weights = [p for name, p in model.named_parameters() if "weight" in name]

      if reg_type == "L1":
          return LAMBDA_L1 * sum(p.abs().sum() for p in weights)

      if reg_type == "L2":
          return LAMBDA_L2 * sum(p.pow(2).sum() for p in weights)

      if reg_type == "ElasticNet":
          l1 = LAMBDA_L1 * sum(p.abs().sum() for p in weights)
          l2 = LAMBDA_L2 * sum(p.pow(2).sum() for p in weights)
          return l1 + l2

      raise ValueError(f"Unknown reg_type '{reg_type}'. Choose None, 'L1', 'L2', or 'ElasticNet'.")


  reg_label = REG_TYPE if REG_TYPE else "None"
  print(f"Regularization : {reg_label}")
  if REG_TYPE in ("L1", "ElasticNet"):
      print(f"  λ_L1 = {LAMBDA_L1}")
  if REG_TYPE in ("L2", "ElasticNet"):
      print(f"  λ_L2 = {LAMBDA_L2}")
  print()

  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr=LR)
  scheduler = optim.lr_scheduler.ReduceLROnPlateau(
      optimizer, patience=SCHEDULER_PATIENCE, factor=0.5
  )

  loss_history      = []   # total loss  (MSE + penalty)
  mse_history       = []   # data term only
  reg_loss_history  = []   # penalty term only

  for epoch in range(1, EPOCHS + 1):
      # set the NN to training mode
      model.train()
      # set gradient to zero
      optimizer.zero_grad()

      # calculate the loss
      pred     = model(x_train)
      mse_loss = criterion(pred, y_train)
      reg_loss = regularization_loss(model, REG_TYPE)
      loss     = mse_loss + reg_loss

      # calculate the gradient of the loss
      loss.backward()
      # update the parameters based on the loss
      optimizer.step()
      scheduler.step(loss)

      #
      loss_history.append(loss.item())
      mse_history.append(mse_loss.item())
      reg_loss_history.append(reg_loss.item())

      if epoch % 500 == 0:
          print(f"Epoch {epoch:5d}/{EPOCHS} | Total: {loss.item():.6f} | "
                f"MSE: {mse_loss.item():.6f} | Reg ({reg_label}): {reg_loss.item():.6f} | "
                f"LR: {optimizer.param_groups[0]['lr']:.2e}")

  model.eval()
  with torch.no_grad():
      y_pred   = model(x_test)

  test_mse = criterion(y_pred, y_test).item()
  print(f"\nTest MSE: {test_mse:.6f}")

  return y_pred, test_mse





In [ ]:
results={"Reg_type": ["Off", "L1", "L2", "ElasticNet"],
         "test_mse": [],
         "y_pred": []}


for reg_type in results["Reg_type"]:
  y_pred, test_mse = do_training(reg_type)
  results["test_mse"].append(test_mse)
  results["y_pred"].append(y_pred)


In [ ]:
for reg_type, y_pred, test_mse in zip(results["Reg_type"], results["y_pred"], results["test_mse"]):
  plt.plot(x_train.numpy(), y_train.numpy(), "o", label="Training samples")
  plt.plot(x_test.numpy(), y_test.numpy(), "k", linewidth=2, label="True function")
  plt.plot(x_test.numpy(), y_pred.numpy(), linewidth=2, linestyle="-", label="NN prediction")
  #label="Reg_type:{:s} (Test MSE: {:.6f})".format(reg_type, test_mse))
  plt.legend()
  plt.xlabel("x");
  plt.ylabel("f(x)")
  plt.title("Reg_type:{:s} (Test MSE: {:.6f})".format(reg_type, test_mse))
  plt.show()

In [ ]:
# Check if CUDA is available
print(torch.cuda.is_available())  # True = GPU available

# See which device you're on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # 'cuda' or 'cpu'

# Get GPU name (if available)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # e.g. 'NVIDIA GeForce RTX 3090'
    print(torch.cuda.device_count())